In [1]:
import os
import torch 
import torch.nn as nn
from spnc import spnc_anisotropy
import numpy as np
import matplotlib.pyplot as plt
import tqdm as tqdm
import pickle
import spnc_ml as ml


from pathlib import Path

CANDIDATES = [
    
    Path(r"C:\Users\tom\Desktop\Repository"),
    Path(r"C:\Users\Chen\Desktop\Repository"),
]
searchpaths = [p for p in CANDIDATES if p.exists()]

#tuple of repos
repos = ('machine_learning_library',)

from deterministic_mask import fixed_seed_mask, max_sequences_mask
import repo_tools
repo_tools.repos_path_finder(searchpaths, repos)
from single_node_res import single_node_reservoir
import ridge_regression as RR
from linear_layer import *
from mask import binary_mask
from utility import *
from NARMA10 import NARMA10
from datasets.load_TI46_digits import *
import datasets.load_TI46 as TI46
from sklearn.metrics import classification_report


# 构建储层对象
class ReservoirParams:
    def __init__(self, **kwargs):
            # Reservoir parameters 
            self.h = 0.4
            self.theta_H = 90
            self.k_s_0 = 0
            self.phi = 45
            self.beta_prime = 35.13826524755751

            # Network parameters 
            self.Nvirt = 50
            self.m0 = 0.005288612874870094
            self.bias = True
            self.Nwarmup = 0
            self.verbose_repr = False

            self.params = {
                'theta': 0.34142235979698393,
                'gamma': 0.069274461903986,
                'delay_feedback': 0,
                'Nvirt': self.Nvirt,
                'length_warmup': self.Nwarmup,
                'warmup_sample': self.Nwarmup * self.Nvirt,
                'voltage_noise': False,
                'seed_voltage_noise': 1234,
                'delta_V': 0.1,
                'johnson_noise': False,
                'seed_johnson_noise': 1234,
                'mean_johnson_noise': 0.0000,
                'std_johnson_noise': 0.00001,
                'thermal_noise': False,
                'seed_thermal_noise': 1234,
                'lambda_ou': 1.0,
                'sigma_ou': 0.1
        }

            for key in ['h', 'theta_H', 'k_s_0', 'phi', 'beta_prime', 'Nvirt', 'm0', 'bias', 'Nwarmup']:
                if key in kwargs:
                    setattr(self, key, kwargs[key])

            
            if 'params' in kwargs and isinstance(kwargs['params'], dict):
                self.params.update(kwargs['params'])

    
    def update_params(self, **kwargs):
        for key, value in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, value)
            if key in self.params:
                self.params[key] = value
            if not hasattr(self, key) and key not in self.params:
                raise AttributeError(f"ReservoirParams has no attribute or param key '{key}'")
            
    def print_params(self, verbose=False):
        if not verbose:
            print(f"ReservoirParams(h={self.h}, beta_prime={self.beta_prime}, Nvirt={self.Nvirt})")
        else:
            print(f"ReservoirParams detailed info:")
            print(f"  h = {self.h}")
            print(f"  theta_H = {self.theta_H}")
            print(f"  k_s_0 = {self.k_s_0}")
            print(f"  phi = {self.phi}")
            print(f"  beta_prime = {self.beta_prime}")
            print(f"  Nvirt = {self.Nvirt}")
            print(f"  m0 = {self.m0}")
            print(f"  bias = {self.bias}")
            print("  params dictionary:")
            for k, v in self.params.items():
                print(f"    {k}: {v}")



In [ ]:
params = ReservoirParams(
        h=0.4, m0=0.008, Nvirt=20, beta_prime=	50.0,
        params={'theta': 0.1564938388583194, 'gamma': 0.04608425844940916, 'Nvirt': 20}
    )
# params = ReservoirParams()

spn_bestMC_test = spnc_anisotropy(
        params.h,
        params.theta_H,
        params.k_s_0,
        params.phi,
        params.beta_prime,
        restart=True
    )

# transform = spn.gen_signal_slow_delayed_feedback
transform_MC_test = spn_bestMC_test.gen_signal_slow_delayed_feedback

speakers = ['f1','f2','f3','f4','f5'] 

acc = ml.spnc_TI46(speakers, params.Nvirt, params.m0, params.bias, transform_MC_test, params.params)
print(acc)

In [3]:
params_50 = ReservoirParams(
        h=0.4, m0=0.05165540820180517, Nvirt=20, beta_prime=	50.0,
        params={'theta': 0.2, 'gamma': 0.04802979116381437, 'Nvirt': 20}
    )
# params = ReservoirParams()

spn_50 = spnc_anisotropy(
        params_50.h,
        params_50.theta_H,
        params_50.k_s_0,
        params_50.phi,
        params_50.beta_prime,
        restart=True
    )

# transform = spn.gen_signal_slow_delayed_feedback
transform_50 = spn_50.gen_signal_slow_delayed_feedback

speakers = ['f1','f2','f3','f4','f5'] 

acc = ml.spnc_TI46(speakers, params_50.Nvirt, params_50.m0, params_50.bias, transform_50, params_50.params)
print(acc)

Samples for training:  500
first 5 samples for training:  [array([ 0,  0, -1, ...,  0,  0,  0], shape=(12800,), dtype=int16)
 array([ 0,  0,  0, ..., -1,  0,  0], shape=(11520,), dtype=int16)
 array([-1, -1,  0, ...,  0,  0,  1], shape=(16128,), dtype=int16)
 array([-1,  0, -1, ..., -1, -1, -1], shape=(19456,), dtype=int16)
 array([-4, -1,  1, ...,  1,  0,  0], shape=(17152,), dtype=int16)]
Samples for test:  795
Using MFCC preprocessing
Nin = 13 , Nout =  10 , Nvirt =  20
Deterministic mask will be used
the shape of the mask is:  (20, 13)
Seed Training: 1234
len(x_train): 7999
error with zero =  0.07927023856870023
Optimal regression parameter =  2.4596031111568544
Train report
              precision    recall  f1-score   support

           0      1.000     0.750     0.857        40
           1      0.729     0.875     0.795        40
           2      0.686     0.875     0.769        40
           3      0.812     0.650     0.722        40
           4      0.878     0.900     0.8

In [4]:
params_33 = ReservoirParams(
        h=0.4, m0=0.17389499595223099, Nvirt=20, beta_prime=	50.0,
        params={'theta': 0.2, 'gamma': 0.09431886362115637, 'Nvirt': 20}
    )
# params = ReservoirParams()

spn_33 = spnc_anisotropy(
        params_33.h,
        params_33.theta_H,
        params_33.k_s_0,
        params_33.phi,
        params_33.beta_prime,
        restart=True
    )

# transform = spn.gen_signal_slow_delayed_feedback
transform_33 = spn_33.gen_signal_slow_delayed_feedback

speakers = ['f1','f2','f3','f4','f5'] 

acc = ml.spnc_TI46(speakers, params_33.Nvirt, params_33.m0, params_33.bias, transform_33, params_33.params)
print(acc)

Samples for training:  500
first 5 samples for training:  [array([ 0,  0, -1, ...,  0,  0,  0], shape=(12800,), dtype=int16)
 array([ 0,  0,  0, ..., -1,  0,  0], shape=(11520,), dtype=int16)
 array([-1, -1,  0, ...,  0,  0,  1], shape=(16128,), dtype=int16)
 array([-1,  0, -1, ..., -1, -1, -1], shape=(19456,), dtype=int16)
 array([-4, -1,  1, ...,  1,  0,  0], shape=(17152,), dtype=int16)]
Samples for test:  795
Using MFCC preprocessing
Nin = 13 , Nout =  10 , Nvirt =  20
Deterministic mask will be used
the shape of the mask is:  (20, 13)
Seed Training: 1234
len(x_train): 7999
error with zero =  0.08088813065292036
Optimal regression parameter =  2.4596031111568544
Train report
              precision    recall  f1-score   support

           0      1.000     0.675     0.806        40
           1      0.702     0.825     0.759        40
           2      0.921     0.875     0.897        40
           3      0.786     0.825     0.805        40
           4      0.796     0.975     0.8

c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

In [5]:
params_367 = ReservoirParams(
        h=0.4, m0=0.06011267415449804, Nvirt=20, beta_prime=	50.0,
        params={'theta': 0.2, 'gamma': 0.05784302397257039, 'Nvirt': 20}
    )
# params = ReservoirParams()

spn_367 = spnc_anisotropy(
        params_367.h,
        params_367.theta_H,
        params_367.k_s_0,
        params_367.phi,
        params_367.beta_prime,
        restart=True
    )

# transform = spn.gen_signal_slow_delayed_feedback
transform_367 = spn_367.gen_signal_slow_delayed_feedback

speakers = ['f1','f2','f3','f4','f5'] 

acc = ml.spnc_TI46(speakers, params_367.Nvirt, params_367.m0, params_367.bias, transform_367, params_367.params)
print(acc)

Samples for training:  500
first 5 samples for training:  [array([ 0,  0, -1, ...,  0,  0,  0], shape=(12800,), dtype=int16)
 array([ 0,  0,  0, ..., -1,  0,  0], shape=(11520,), dtype=int16)
 array([-1, -1,  0, ...,  0,  0,  1], shape=(16128,), dtype=int16)
 array([-1,  0, -1, ..., -1, -1, -1], shape=(19456,), dtype=int16)
 array([-4, -1,  1, ...,  1,  0,  0], shape=(17152,), dtype=int16)]
Samples for test:  795
Using MFCC preprocessing
Nin = 13 , Nout =  10 , Nvirt =  20
Deterministic mask will be used
the shape of the mask is:  (20, 13)
Seed Training: 1234
len(x_train): 7999
error with zero =  0.07986964622515302
Optimal regression parameter =  2.4596031111568544
Train report
              precision    recall  f1-score   support

           0      1.000     0.725     0.841        40
           1      0.705     0.775     0.738        40
           2      0.795     0.875     0.833        40
           3      0.939     0.775     0.849        40
           4      0.795     0.875     0.8

In [6]:
params_247 = ReservoirParams(
        h=0.4, m0=0.0833349841620083, Nvirt=200, beta_prime=	50.0,
        params={'theta': 0.2, 'gamma': 0.06423651459901063, 'Nvirt': 200}
    )
# params = ReservoirParams()

spn_247 = spnc_anisotropy(
        params_247.h,
        params_247.theta_H,
        params_247.k_s_0,
        params_247.phi,
        params_247.beta_prime,
        restart=True
    )

# transform = spn.gen_signal_slow_delayed_feedback
transform_247 = spn_247.gen_signal_slow_delayed_feedback

speakers = ['f1','f2','f3','f4','f5'] 

acc = ml.spnc_TI46(speakers, params_247.Nvirt, params_247.m0, params_247.bias, transform_247, params_247.params)
print(acc)

Samples for training:  500
first 5 samples for training:  [array([ 0,  0, -1, ...,  0,  0,  0], shape=(12800,), dtype=int16)
 array([ 0,  0,  0, ..., -1,  0,  0], shape=(11520,), dtype=int16)
 array([-1, -1,  0, ...,  0,  0,  1], shape=(16128,), dtype=int16)
 array([-1,  0, -1, ..., -1, -1, -1], shape=(19456,), dtype=int16)
 array([-4, -1,  1, ...,  1,  0,  0], shape=(17152,), dtype=int16)]
Samples for test:  795
Using MFCC preprocessing
Nin = 13 , Nout =  10 , Nvirt =  200
Deterministic mask will be used
the shape of the mask is:  (200, 13)
Seed Training: 1234
len(x_train): 7999
error with zero =  0.059786932411771354
Optimal regression parameter =  2.4596031111568544
Train report
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        40
           1      0.976     1.000     0.988        40
           2      1.000     1.000     1.000        40
           3      1.000     1.000     1.000        40
           4      1.000     1.000     

c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

In [ ]:
params_2471 = ReservoirParams(
        h=0.4, m0=0.0833349841620083, Nvirt=20, beta_prime=	50.0,
        params={'theta': 0.2, 'gamma': 0.06423651459901063, 'Nvirt': 20}
    )
# params = ReservoirParams()

spn_2471 = spnc_anisotropy(
        params_2471.h,
        params_2471.theta_H,
        params_2471.k_s_0,
        params_2471.phi,
        params_2471.beta_prime,
        restart=True
    )

# transform = spn.gen_signal_slow_delayed_feedback
transform_2471 = spn_2471.gen_signal_slow_delayed_feedback

speakers = ['f1','f2','f3','f4','f5'] 

acc = ml.spnc_TI46(speakers, params_2471.Nvirt, params_2471.m0, params_2471.bias, transform_2471, params_2471.params)
print(acc)

Samples for training:  500
first 5 samples for training:  [array([ 0,  0, -1, ...,  0,  0,  0], shape=(12800,), dtype=int16)
 array([ 0,  0,  0, ..., -1,  0,  0], shape=(11520,), dtype=int16)
 array([-1, -1,  0, ...,  0,  0,  1], shape=(16128,), dtype=int16)
 array([-1,  0, -1, ..., -1, -1, -1], shape=(19456,), dtype=int16)
 array([-4, -1,  1, ...,  1,  0,  0], shape=(17152,), dtype=int16)]
Samples for test:  795
Using MFCC preprocessing
Nin = 13 , Nout =  10 , Nvirt =  20
Deterministic mask will be used
the shape of the mask is:  (20, 13)
Seed Training: 1234
len(x_train): 7999
error with zero =  0.08030843736957502
Optimal regression parameter =  2.4596031111568544
Train report
              precision    recall  f1-score   support

           0      0.964     0.675     0.794        40
           1      0.740     0.925     0.822        40
           2      0.738     0.775     0.756        40
           3      0.886     0.775     0.827        40
           4      0.946     0.875     0.9

我很好奇Nv和Ti46acc之间的函数关系

In [4]:
# create a list of Nv
Nv_list =[20,50,100,150,200]

speakers = ['f1','f2','f3','f4','f5'] 

# create a dictionary to store the results
result_dict = {}

# loop through the list of Nv
for idx, Nv in enumerate(Nv_list):
    # create a new ReservoirParams
    params = ReservoirParams(
        h=0.4, m0=0.0833349841620083, Nvirt=Nv, beta_prime=50.0,
        params={'theta': 0.2, 'gamma': 0.06423651459901063, 'Nvirt': Nv}
    )
    # create a new spnc_anisotropy
    spn = spnc_anisotropy(
        params.h,
        params.theta_H,
        params.k_s_0,
        params.phi,
        params.beta_prime,
        restart=True
    )
    # set the transform
    transform = spn.gen_signal_slow_delayed_feedback
    # perform the TI46 task
    acc = ml.spnc_TI46(speakers, params.Nvirt, params.m0, params.bias, transform, params.params)

    # store the result
    result_dict[idx] =  {
        'Nv': Nv,
        'acc': acc
    }
    print(f"Virtual nodes: {Nv}, TI46 accuracy: {acc}")



    

Samples for training:  500
first 5 samples for training:  [array([ 0,  0, -1, ...,  0,  0,  0], shape=(12800,), dtype=int16)
 array([ 0,  0,  0, ..., -1,  0,  0], shape=(11520,), dtype=int16)
 array([-1, -1,  0, ...,  0,  0,  1], shape=(16128,), dtype=int16)
 array([-1,  0, -1, ..., -1, -1, -1], shape=(19456,), dtype=int16)
 array([-4, -1,  1, ...,  1,  0,  0], shape=(17152,), dtype=int16)]
Samples for test:  795
Using MFCC preprocessing
Nin = 13 , Nout =  10 , Nvirt =  20
Deterministic mask will be used
the shape of the mask is:  (20, 13)
Seed Training: 1234
len(x_train): 8000
error with zero =  0.08037443428025393
Optimal regression parameter =  2.4596031111568544
Train report
              precision    recall  f1-score   support

           0      1.000     0.650     0.788        40
           1      0.729     0.875     0.795        40
           2      0.811     0.750     0.779        40
           3      0.925     0.925     0.925        40
           4      0.902     0.925     0.9

c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

              precision    recall  f1-score   support

           0      0.000     0.000     0.000        75
           1      0.000     0.000     0.000        80
           2      0.101     1.000     0.183        80
           3      0.000     0.000     0.000        80
           4      0.000     0.000     0.000        80
           5      0.000     0.000     0.000        80
           6      0.000     0.000     0.000        80
           7      0.000     0.000     0.000        80
           8      0.000     0.000     0.000        80
           9      0.000     0.000     0.000        80

    accuracy                          0.101       795
   macro avg      0.010     0.100     0.018       795
weighted avg      0.010     0.101     0.018       795

Virtual nodes: 50, TI46 accuracy: 0.10062893081761007
Samples for training:  500
first 5 samples for training:  [array([ 0,  0, -1, ...,  0,  0,  0], shape=(12800,), dtype=int16)
 array([ 0,  0,  0, ..., -1,  0,  0], shape=(11520,), dtype=in

c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chen\.conda\envs\spnc_python_3_12\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",